# 11 — Reliability Score Weight Sensitivity (Python + R, one notebook)

**Continuation of notebooks 01–06.** Part A (Python) builds 4 alternative
reliability score weightings from the same underlying signals. Part B (R)
re-runs the `faircause` causal estimation once per weighting, so you can
see whether the causal conclusion is stable or fragile with respect to
the heuristic 0.6/0.2/0.2 choice.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
df = pd.read_csv("metr_la_metrics.csv")
n = len(df)
df.head()

In [ ]:
# --- Build 4 alternative reliability score versions ---

# (a) Original heuristic weighting
df["reliability_original"] = 1 - (0.6*df.zero_rate + 0.2*df.cusum_flag_rate + 0.2*df.ewma_flag_rate)

# (b) Equal weighting
df["reliability_equal"] = 1 - (df.zero_rate + df.cusum_flag_rate + df.ewma_flag_rate) / 3

# (c) Zero-rate only
df["reliability_zero_only"] = 1 - df.zero_rate

# (d) PCA-derived weights (data-driven, not heuristic)
signals = df[["zero_rate", "cusum_flag_rate", "ewma_flag_rate"]].values
signals_std = (signals - signals.mean(axis=0)) / signals.std(axis=0)
pca = PCA(n_components=1)
pc1 = pca.fit_transform(signals_std).flatten()
# orient so higher = less reliable, matching the other scores' direction
if np.corrcoef(pc1, df.zero_rate)[0,1] < 0:
    pc1 = -pc1
pc1_norm = (pc1 - pc1.min()) / (pc1.max() - pc1.min())
df["reliability_pca"] = 1 - pc1_norm
print("PCA loadings (zero_rate, cusum_flag_rate, ewma_flag_rate):", pca.components_[0])

df[["node_id", "reliability_original", "reliability_equal",
    "reliability_zero_only", "reliability_pca"]].describe()


In [ ]:
# --- Correlation between the 4 versions, so you know upfront how different
#     they actually are before running the (much slower) R re-estimation ---
corr = df[["reliability_original", "reliability_equal",
           "reliability_zero_only", "reliability_pca"]].corr()
print(corr.round(3))
print("\nIf these are all >0.9 correlated with each other, the R re-estimation")
print("in the next file will likely show a stable causal conclusion regardless")
print("of weighting. If any pair is well below 0.9, expect the causal estimate")
print("to meaningfully shift between weighting schemes.")


In [ ]:
# traffic_regime, road_type, density, topology are already in df
df["disparity"] = df["persistence_error"]

df.to_csv("reliability_variants.csv", index=False)
print("Saved reliability_variants.csv")

## Part B — R: re-run `faircause` for each weighting scheme

Consumes `reliability_variants.csv` written by Part A above.

In [ ]:
# --- R environment setup for this notebook ---
import subprocess
import sys
subprocess.run(["apt-get", "install", "-y", "-qq", "r-base-core"], stdout=subprocess.DEVNULL)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rpy2"])
get_ipython().run_line_magic("load_ext", "rpy2.ipython")
print("R + rpy2 bridge ready.")


In [ ]:
%%R
if (!requireNamespace("devtools", quietly = TRUE)) install.packages("devtools")
if (!requireNamespace("faircause", quietly = TRUE)) devtools::install_github("dplecko/CFA")
library(faircause)
cat("faircause loaded, version:", as.character(packageVersion("faircause")), "\n")


In [ ]:
%%R
#!/usr/bin/env Rscript
# ============================================================================
# 04_reliability_sensitivity_R.R
#
# R half of the reliability weight sensitivity check. Consumes
# reliability_variants.csv from 04_reliability_sensitivity_PYTHON.ipynb.
# NOT executed/tested locally. Run in Colab after the Python half.
# ============================================================================

library(faircause)

data <- read.csv("reliability_variants.csv")
data$density_bin <- ifelse(data$density > median(data$density),
                            "high_density", "low_density")

weight_schemes <- c("reliability_original", "reliability_equal",
                     "reliability_zero_only", "reliability_pca")

results_table <- data.frame()

for (scheme in weight_schemes) {
  cat("\n=== Running faircause with:", scheme, "===\n")

  df_run <- data
  df_run$reliability_active <- df_run[[scheme]]

  result <- tryCatch({
    fairness_cookbook(
      data = df_run,
      X = "density_bin",
      Z = c("traffic_regime", "road_type"),
      W = c("reliability_active", "topology"),
      Y = "disparity",
      x0 = "low_density", x1 = "high_density"
    )
  }, error = function(e) {
    cat("ERROR for scheme", scheme, ":", conditionMessage(e), "\n")
    NULL
  })

  if (!is.null(result)) {
    s <- as.data.frame(summary(result))
    s$weight_scheme <- scheme
    results_table <- rbind(results_table, s)
  }
}

cat("\n=== Comparison table across weighting schemes ===\n")
print(results_table)

write.csv(results_table, "reliability_sensitivity_results.csv", row.names = FALSE)
cat("\nSaved reliability_sensitivity_results.csv\n")

cat("\n=== Interpretation ===\n")
cat("Look at the Ctf-IE for 'reliability_active' across the 4 rows for each\n")
cat("weight scheme. If the sign and rough magnitude are consistent across all\n")
cat("4, the causal conclusion is robust to the heuristic weight choice. If it\n")
cat("flips sign or changes by more than ~50% between schemes, the original\n")
cat("0.6/0.2/0.2 weighting was not a safe arbitrary choice and needs either\n")
cat("(a) justification beyond 'it seemed reasonable', or (b) replacement with\n")
cat("the PCA-derived (data-driven) version as the primary reported result.\n")

cat("\nNOTE: as in 03_power_simulation.R, the exact column names in\n")
cat("summary(result) need to be confirmed once you've actually run\n")
cat("02_ctf_estimation_faircause.R and inspected the real output structure —\n")
cat("adjust the code above if `as.data.frame(summary(result))` doesn't match\n")
cat("what's assumed here.\n")